# 12. XGBoost Training (4개 모델)

## 목적
4개 예측 타겟에 대해 XGBoost 모델 학습 및 평가

## 예측 타겟
| 모델 | 타겟 | 예측 시점 | 임상적 의미 |
|------|------|----------|------------|
| Mortality | death_next_24h | 24시간 내 사망 | 전반적 예후 |
| Ventilator | vent_next_12h | 12시간 내 인공호흡기 | 호흡 악화 |
| Vasopressor | pressor_next_12h | 12시간 내 승압제 | 순환 악화 |
| Composite | composite_next_24h | 24시간 내 악화 | 종합 악화 |

## 입력
- `train.csv`, `val.csv`, `test.csv`
- `feature_cols.json`
- `scale_pos_weights.json`

## 출력
- 학습된 모델 (`.json`)
- 성능 결과 (`results_*.json`, `results_*.csv`)
- 시각화 (ROC, PR curves)

## Step 3: Vital Signs 결측치 처리

* 대상: hr, rr, spo2, temp, sbp, dbp, mbp
    * 결측률: 1~4% (매우 낮음)

* 전략: FFill → Median
    * ICU에서 거의 hourly 모니터링 → 결측 = 센서 일시 탈착 수준
    * 직전 값으로 채우는 것이 임상적으로 합리적
    * 결측률 낮아서 복잡한 처리 불필요

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import xgboost as xgb
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 성능 평가 모듈
from metrics import (
    evaluate_model, print_summary, print_confusion_matrices,
    plot_roc_curves, plot_pr_curves, plot_combined_roc,
    save_results, plot_feature_importance
)

print("=== 02. XGBoost Training 시작 ===")
print(f"XGBoost version: {xgb.__version__}")

## 설정

In [ ]:
# ==============================================================================
# 설정
# ==============================================================================

# 데이터 경로 (11_model_prep에서 생성된 폴더)
DATA_DIR = '../../data-pipeline/data/processed/all_features_v3'  # 또는 'top20', 'custom_features' 등
OUTPUT_DIR = '../models/xgboost'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 예측 타겟 정의
TARGETS = {
    'death': 'death_next_24h',        # 24시간 내 사망
    'vent': 'vent_next_12h',          # 12시간 내 인공호흡기
    'pressor': 'pressor_next_12h',    # 12시간 내 승압제
    'composite': 'composite_next_24h'  # 24시간 내 종합 악화
}

# XGBoost 기본 하이퍼파라미터
BASE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'aucpr'],
    'max_depth': 6,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'gamma': 0,
    'reg_alpha': 0,
    'reg_lambda': 1,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0
}

# Early Stopping
EARLY_STOPPING_ROUNDS = 50
N_ESTIMATORS = 1000
VERSION = 'v3'
os.makedirs(f'{OUTPUT_DIR}/{VERSION}', exist_ok=True)

print(f"데이터 경로: {DATA_DIR}")
print(f"모델 저장 경로: {OUTPUT_DIR}")

## Step 1: 데이터 로드

In [ ]:
print("\nStep 1: 데이터 로드")

# CSV 로드
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"✓ Train: {len(train):,} rows")
print(f"✓ Val: {len(val):,} rows")
print(f"✓ Test: {len(test):,} rows")

# 피처 목록 로드
with open(os.path.join(DATA_DIR, 'feature_cols.json'), 'r') as f:
    feature_cols = json.load(f)
print(f"✓ 피처: {len(feature_cols)}개")

# 클래스 가중치 로드
with open(os.path.join(DATA_DIR, 'scale_pos_weights.json'), 'r') as f:
    scale_pos_weights = json.load(f)
print(f"✓ 클래스 가중치 로드 완료")

In [ ]:
# X, y 분리
X_train = train[feature_cols]
X_val = val[feature_cols]
X_test = test[feature_cols]

print(f"\n피처 shape:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")

# 레이블 분포 확인
print(f"\n=== 타겟 레이블 분포 (Train) ===")
for name, col in TARGETS.items():
    pos_rate = train[col].mean() * 100
    pos_count = train[col].sum()
    weight = scale_pos_weights.get(col, 1)
    print(f"  {name}: {pos_count:,} ({pos_rate:.2f}%) | scale_pos_weight={weight:.1f}")

## Step 2: 모델 학습 (4개 타겟)

### 클래스 불균형 대응
- `scale_pos_weight`: Positive 클래스에 가중치 부여
- Early Stopping: Validation AUROC 기준

### 학습 전략
- 각 타겟별 별도 모델 학습
- Validation set으로 Early Stopping

In [ ]:
print("\nStep 2: 모델 학습")

models = {}
training_history = {}

for name, target_col in TARGETS.items():
    print(f"\n{'='*50}")
    print(f"[{name.upper()}] Training: {target_col}")
    print(f"{'='*50}")
    
    # 레이블 추출
    y_train = train[target_col]
    y_val = val[target_col]
    
    # 클래스 가중치
    weight = scale_pos_weights.get(target_col, 1)
    print(f"  scale_pos_weight: {weight:.1f}")
    
    # 파라미터 설정
    params = BASE_PARAMS.copy()
    params['scale_pos_weight'] = weight
    
    # 모델 생성 및 학습
    model = xgb.XGBClassifier(
        **params,
        n_estimators=N_ESTIMATORS,
        tree_method='hist',
        early_stopping_rounds=EARLY_STOPPING_ROUNDS
    )
    
    # 학습
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    
    # 결과 저장
    models[name] = model
    best_iteration = model.best_iteration
    
    # 학습 히스토리
    results = model.evals_result()
    training_history[name] = results
    
    print(f"  ✓ Best iteration: {best_iteration}")
    print(f"  ✓ Train AUC: {results['validation_0']['auc'][best_iteration]:.4f}")
    print(f"  ✓ Val AUC: {results['validation_1']['auc'][best_iteration]:.4f}")

print(f"\n{'='*50}")
print("✓ 모든 모델 학습 완료!")
print(f"{'='*50}")

## Step 3: 모델 평가 (Test Set)

In [ ]:
import numpy as np
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

print("\nStep 3: 모델 평가 (Test Set) - Multi Threshold Sweep")

# 보고 싶은 임계값들 (원하는 대로 수정)
THRESH_LIST = np.round(np.arange(0.01, 0.51, 0.01), 2).tolist()

results = {}      # 모델별 threshold 결과 저장
summary = {}      # 모델별 AUROC/AUPRC 저장

for name, col in TARGETS.items():
    y_true = test[col].values
    y_prob = models[name].predict_proba(X_test)[:, 1]

    # threshold 무관 지표는 1번만
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    summary[name] = {"auroc": auroc, "auprc": auprc}

    # threshold sweep
    rows = []
    for thr in THRESH_LIST:
        y_pred = (y_prob >= thr).astype(int)

        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        rows.append({
            "threshold": thr,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "specificity": spec,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn
        })

    results[name] = rows

# ===== 출력 =====
print("=" * 110)
print(" XGBoost v2 Performance (Threshold Sweep for ALL 4 Models)")
print("=" * 110)

for name in TARGETS.keys():
    auroc = summary[name]["auroc"]
    auprc = summary[name]["auprc"]

    print(f"\n[Model: {name}]  AUROC={auroc:.4f}  AUPRC={auprc:.4f}")
    print("-" * 110)
    print(f"{'Thresh':>6} {'Prec':>8} {'Recall':>8} {'F1':>8} {'Spec':>8}  {'TP':>7} {'FP':>7} {'FN':>7} {'TN':>7}")
    print("-" * 110)

    for r in results[name]:
        print(f"{r['threshold']:>6.2f} {r['precision']:>8.4f} {r['recall']:>8.4f} {r['f1']:>8.4f} {r['specificity']:>8.4f}  "
              f"{r['tp']:>7} {r['fp']:>7} {r['fn']:>7} {r['tn']:>7}")

    print("-" * 110)


In [ ]:
# Recall ≥ 0.80 중에서 F1 최대
MIN_RECALL = 0.80

print("\n Best Threshold per Model (Recall ≥ 0.80 & Max F1)")
print("=" * 80)
print(f"{'Model':<12} {'Thresh':>6} {'F1':>8} {'Prec':>8} {'Recall':>8} {'Spec':>8}")

for name, rows in results.items():
    candidates = [r for r in rows if r["recall"] >= MIN_RECALL]

    if not candidates:
        print(f"{name:<12}  ⚠️ No threshold meets recall ≥ {MIN_RECALL}")
        continue

    best = max(candidates, key=lambda x: x["f1"])

    print(f"{name:<12} {best['threshold']:>6.2f} "
          f"{best['f1']:>8.4f} {best['precision']:>8.4f} "
          f"{best['recall']:>8.4f} {best['specificity']:>8.4f}")


In [ ]:
# [Detection Summary – Recall ≥ MIN_RECALL & Max F1]
MIN_RECALL = 0.80

print("\n[Detection Summary – Recall ≥ {:.2f} & Max F1]".format(MIN_RECALL))
print(f"{'Model':<12} {'Thresh':>6} {'F1':>8} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'Caught%':>10}")
print("-" * 80)

for name, rows in results.items():
    # 1) recall 조건 만족하는 후보만
    candidates = [r for r in rows if r.get("recall", 0.0) >= MIN_RECALL]

    # 2) 없으면 경고 출력
    if not candidates:
        print(f"{name:<12}  {'-':>6} {'-':>8} {'-':>8} {'-':>8} {'-':>8} {'-':>8}  ⚠️ No threshold meets recall ≥ {MIN_RECALL}")
        continue

    # 3) 후보 중 F1 최대 선택 (tie-breaker: threshold 낮은 쪽 선호 등 원하면 추가 가능)
    best = max(candidates, key=lambda x: x.get("f1", 0.0))

    thr = best.get("threshold", None)
    f1  = best.get("f1", 0.0)

    tp = best.get("tp", 0)
    fp = best.get("fp", 0)
    fn = best.get("fn", 0)
    tn = best.get("tn", 0)

    total_pos = tp + fn
    caught_pct = (tp / total_pos * 100) if total_pos > 0 else 0.0

    print(f"{name:<12} {thr:>6.2f} {f1:>8.4f} {tp:>8} {fp:>8} {fn:>8} {tn:>8} {caught_pct:>9.1f}%")


In [ ]:
print("\n[Detection Summary – Best F1 Threshold]")
print(f"{'Model':<12} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'Caught%':>10}")
print("-" * 60)

for name, rows in results.items():
    best = max(rows, key=lambda x: x["f1"])

    tp, fp, fn, tn = best["tp"], best["fp"], best["fn"], best["tn"]
    total_pos = tp + fn
    caught_pct = tp / total_pos * 100 if total_pos > 0 else 0.0

    print(f"{name:<12} {tp:>8} {fp:>8} {fn:>8} {tn:>8} {caught_pct:>9.1f}%")

In [ ]:
print("\n" + "=" * 80)
print(" Best Threshold per Model (Max F1)")
print("=" * 80)
print(f"{'Model':<12} {'Thresh':>6} {'F1':>8} {'Prec':>8} {'Recall':>8} {'Spec':>8}")
print("-" * 80)

best_thresholds = {}

for name, rows in results.items():
    # F1 최대인 row 선택
    best = max(rows, key=lambda x: x["f1"])
    best_thresholds[name] = best["threshold"]

    print(f"{name:<12} {best['threshold']:>6.2f} "
          f"{best['f1']:>8.4f} {best['precision']:>8.4f} "
          f"{best['recall']:>8.4f} {best['specificity']:>8.4f}")

print("-" * 80)


In [ ]:
print("\n[Detection Summary – Recall ≥ {:.2f} & Max F1]".format(MIN_RECALL))
print(f"{'Model':<12} {'TP':>8} {'FP':>8} {'FN':>8} {'TN':>8} {'Caught%':>10}")
print("-" * 60)

for name, rows in results.items():
    best = max(rows, key=lambda x: x["f1"])

    tp, fp, fn, tn = best["tp"], best["fp"], best["fn"], best["tn"]
    total_pos = tp + fn
    caught_pct = tp / total_pos * 100 if total_pos > 0 else 0.0

    print(f"{name:<12} {tp:>8} {fp:>8} {fn:>8} {tn:>8} {caught_pct:>9.1f}%")


## Step 4: 모델 저장

In [ ]:
print("\nStep 6: 모델 저장")

import pickle

# 시간대별 디렉토리 생성
time_windows = {
    'death': '24h',
    'vent': '12h', 
    'pressor': '12h',
    'composite': '24h'
}

# 결과 저장 전에 numpy 타입 변환
import numpy as np
import pandas as pd
from datetime import datetime, date

def convert_to_native(obj):
    """numpy/pandas 타입을 JSON 저장 가능한 Python 기본 타입으로 변환"""
    # dict
    if isinstance(obj, dict):
        return {str(k): convert_to_native(v) for k, v in obj.items()}

    # list/tuple/set  ✅ 이게 빠져있어서 터진 거임
    if isinstance(obj, (list, tuple, set)):
        return [convert_to_native(v) for v in obj]

    # numpy scalar (np.int64, np.float64, np.bool_, ...)
    if isinstance(obj, np.generic):
        return obj.item()

    # numpy array
    if isinstance(obj, np.ndarray):
        return obj.tolist()

    # pandas types (혹시 들어있을 때 대비)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()

    # datetime/date
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()

    return obj


# 각 모델 저장
for name, model in models.items():
    time_window = time_windows[name]
    
    # 경로: xgboost/v1/24h/death.pkl
    model_dir = os.path.join(OUTPUT_DIR, VERSION, time_window)
    os.makedirs(model_dir, exist_ok=True)
    
    model_path = os.path.join(model_dir, f'{name}.pkl')
    
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    
    print(f"  ✓ {name}: {model_path}")


# 결과 저장 (버전 폴더에)
results_dir = os.path.join(OUTPUT_DIR, VERSION)

# 변환 후 저장
results_native = convert_to_native(results)
save_results(results_native, results_dir, prefix='xgboost_')

In [ ]:
THRESHOLDS = {
    "death": {
        "threshold": 0.48,
        "criterion": "recall>=0.80_max_f1",
        "f1": 0.1527,
        "precision": 0.0844,
        "recall": 0.8019,
        "specificity": 0.8631
    },
    "vent": {
        "threshold": 0.32,
        "criterion": "recall>=0.80_max_f1",
        "f1": 0.0391,
        "precision": 0.0200,
        "recall": 0.8064,
        "specificity": 0.5701
    },
    "pressor": {
        "threshold": 0.38,
        "criterion": "recall>=0.80_max_f1",
        "f1": 0.0457,
        "precision": 0.0235,
        "recall": 0.8033,
        "specificity": 0.6464
    },
    "composite": {
        "threshold": 0.36,
        "criterion": "recall>=0.80_max_f1",
        "f1": 0.1442,
        "precision": 0.0792,
        "recall": 0.8001,
        "specificity": 0.5793
    }
}


In [ ]:
# 학습 설정 저장 (thresholds 추가)
config = {
    'model_type': 'XGBoost',
    'version': VERSION,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'data_dir': DATA_DIR,
    'n_features': len(feature_cols),
    'feature_cols': feature_cols,
    'targets': TARGETS,
    'thresholds': THRESHOLDS,  # 추가
    'params': BASE_PARAMS,
    'n_estimators': N_ESTIMATORS,
    'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
    'best_iterations': {name: model.best_iteration for name, model in models.items()},
    'results': results
}

# configs/xgboost/v2/ 경로에 저장
config_dir = os.path.join('../configs/xgboost', VERSION)
os.makedirs(config_dir, exist_ok=True)

config_path = os.path.join(config_dir, 'training_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f"\n✓ Training config saved: {config_path}")